# Quick start
* `streamlit run interface/demo.py`
* See also `./interface/README.md`

# 1. Data cleaning and extraction
* We put the downloaded corpus into `./data/xml`
* In `./data_prepare`, we coded a `CorpusCleaner` class to clean data
* See `./data_prepare/data_prepare.ipynb` for reproduction

## 1.1 Extract t9-pinyin-word tuple
* In a few words, we extracted the words in form `('52944', 'lazhi', '蜡纸')`. The pinyin and words are provided by the dataset. The t9 is provided by our `T9PinyinHanziConverter` class in `./tools/t9_hanzi_converter`.
* We put our data into `./data/zh_T9_dataset`.But in the end, we did not use this dataset to train our model

## 1.2 Extract sentences
* We extracted all sentences from xml files in `./data/xml/character/`. These files are tokenized sentences of literature works.
* We put extracted sentences into `./data/zh_sent_dataset.tsv`. We use these data to train our model.

## 1.3 Commentary
* We actually don't need the files in `./data/xml/pinyin`, since `hanzi -> pinyin` relation is a function (one hanzi sequence maps to a unique pinyin sequence, except for some margin cases of polyphonic words). Libraries like `lazy_pinyin` can already achieve great results, and handle most cases of polyphony. `pinyin -> t9` is strictly a function. We can thus use plain Chinese sentences to generate our training data. So any Chinese datasets will do, like wikipedia. These datasets might even have better results as they are closer to everyday language, compared to our current dataset of literature works.
* Since the current dataset is provided to us in the context of this project, we decided to stick with it

# 2. Data generation and augmentation
* We coded a `DataAugmentor` class
* See `./data_prepare/data_augmentation.ipynb` to reproduce the process

## 2.1 Data visualization
* See `./data_prepare/corpus_statistics.ipynb`
* We looked at the average length of a sentence and a t9 sequence, in order to decide the context length of our model
* The average length of a sentence is 21, and the average length of a t9 sequence is 7. We thus chose the context length of our model to be 32.

## 2.2 Example and explanation
* We now have sentences like `云 在 天空 游荡 ， 它 从 远方 飘 来 ， 又 飘 向 远方` We will provide labels and augment the data
* Since the model we train should have knowledge of the context, we transform this sentence into these training examples, listed in the cell below
* For example, if we already have `云在` and `天空` t9 sequence is `84265664`, user can type any numbers like `8`, `84`, `842`... and the model should predict `天空`.
* The model should only predict hanzi, so we ignore all cases with punctuations
* We also ignored single words, i.e. cases where there is no context, when users just start to type. We should also add them into our training data (but we forgot...)
* There are some other minor considerations, like we front-truncate the sentences with max length set to 32, as we don't expect the user to type very long sentences. In fact, these steps are better left to a tokenizer class then a DataAugmentor class
* After augmentation, we have roughly 3.7 million data

## 2.3 Output
* We decided to output data into `./data/training_data/training_corpus.parquet` because `datasets` library uses apache arrow to avoid loading the whole dataset into memory


In [1]:
"""
(['云', '9'], '在')
(['云', '9', '2'], '在')
(['云', '9', '2', '4'], '在')
(['云', '在', '8'], '天空')
(['云', '在', '8', '4'], '天空')
(['云', '在', '8', '4', '2'], '天空')
(['云', '在', '8', '4', '2', '6'], '天空')
(['云', '在', '8', '4', '2', '6', '5'], '天空')
(['云', '在', '8', '4', '2', '6', '5', '6'], '天空')
(['云', '在', '8', '4', '2', '6', '5', '6', '6'], '天空')
(['云', '在', '8', '4', '2', '6', '5', '6', '6', '4'], '天空')
"""

"\n(['云', '9'], '在')\n(['云', '9', '2'], '在')\n(['云', '9', '2', '4'], '在')\n(['云', '在', '8'], '天空')\n(['云', '在', '8', '4'], '天空')\n(['云', '在', '8', '4', '2'], '天空')\n(['云', '在', '8', '4', '2', '6'], '天空')\n(['云', '在', '8', '4', '2', '6', '5'], '天空')\n(['云', '在', '8', '4', '2', '6', '5', '6'], '天空')\n(['云', '在', '8', '4', '2', '6', '5', '6', '6'], '天空')\n(['云', '在', '8', '4', '2', '6', '5', '6', '6', '4'], '天空')\n"

# 3. Tokenizer choice

## 3.1 Our expectation of tokenizer's behaviour
* We want a word-level tokenizer, not a character-level one. The reason is that our dataset is tokenized into words, and we want our model to predict words rather than single characters
* Since we already have our tokenized datasets, we want the tokenizer to learn from them and tokenize new sentences in a similar manner
* Otherwise, the model may not predict the right results. If the model only learned from `云 在 天空 游荡` but our tokenizer gives `云在 天 空游 荡` during deployment, the model will be very confused
* We want our tokenizer to tokenize all digits into single digits, since digits present the input of t9. This logic can be easily hardcoded.

## 3.2 Modern tokenizers
* We firstly tried some modern tokenizers in `tokenizers` library, using BPE, Unigram or other algorithms.
* We lost the `.ipynb` file documenting our process of experimentation...
* We tried some already existing tokenizers, used by models like deepseek or bert-base-chinese. But they don't meet our requirements. They don't tokenize as expected, since most of them are sub-word or character level tokenizer
* We tried to create new tokenizers and fit our datasets to train them. But our datasets are possibly too small for them to learn. At the end they often behave like char-level tokenizer
* In the end, we give up using modern tokenizers and turn to more conventional ones

## 3.3 Custom jieba tokenizer
* We implemented a `JiebaLikeTokenizer` class in `./custom_tokenizers/jieba_tokenizer.py`. The class is implemented in a manner to match the tokenizer interface of HuggingFace tokenizers, with features customised to our use (for example context-length set to 32).
* Jieba tokenizer has the advantage that they can learn from our dataset's feature, based on term frequency
* See `./data_prepare/create_tokenizer_training_data.ipynb` to see how we created term frequency file for the tokenizer
* See `./custom_tokenizers/test_tokenizer.ipynb` to see how JiebaLikeTokenizer behaves
* At the end, this tokenizer tokenizes sentences in almost the exact same way as it is in our dataset. They match quite perfectly that we suspect if the original dataset is already tokenized by jieba...
* The tokenizer has some features to meet the model's input requirements, as described below

## 3.4 Commentary
* The major problem we encountered is that we struggled to find a tokenizer that tokenizes in the same way as the dataset. We can actually use a untokenized dataset an train a tokenizer from scratch, then use our trained tokenizer to tokenize the dataset and to generate training data
* But since the jieba tokenizer worked well and we want to stick with the provided data, we will proceed
* It could be a nice idea to ignore rare words to reduce vocab size.

# Model definition

* Since we want our model to understand context we opted for a transformer model. Both encoder and decoder model would work. The job is essentially a next-token-prediction task and we don't even need the model to be auto-regressive. The idea is to choose a simple architecture. We chose a encoder-only architecture and design a bert-like model.
* Model structures are in `./models/model_definitions`. There are two files, which define the same model with pytorch and tensorflow. We decided to move entirely to pytorch mainly for python version compatibility issues. Also the `transformers` library offers a native BertModel class, where in tensorflow we have to write it manually.

## Hyper-parameter tuning
* See the cell below for the original hyperparameter that we found on [HuggingFace](https://huggingface.co/docs/transformers/en/model_doc/bert), and explanation of our own choice
* See `./training/tuner_setup.py` and `./training/tuner.ipynb` for the actual tuning
* See cell below for results

## Commentary
* The reason we use keras tuner is because we first defined our model using tensorflow. After encountering some issues we decide to switch to pytorch. Rewriting the whole tuning scripts using optuna is maybe a better idea. But we decided to stick with keras tuner since we are more familiar with it

In [1]:
# Our tuning philosophy is simple. We only tune the hyperparameters which affect model size
# We want these hyperparameters to be more or less proportional to the original ones
# For the rest, like dropout rate, we have confidence in the original setting
# Some hyperparameter is predefined, like context length (max_position_embeddings)
# We don't want our model to be too big

vocab_size = 30522  # Fixed to our custom tokenizer's vocab size
hidden_size = 768  # Tunable: [64, 128, 256]
num_hidden_layers = 12  # Tunable: 1 to 4
num_attention_heads = 12  # Tunable: [2, 4, 8]
intermediate_size = 3072  # Tunable: [128, 256, 512]
hidden_act = 'gelu'  # Tunable but safer to stick with it, does not affect model size, in our opinion smoother than relu
hidden_dropout_prob = 0.1  # Tunable but safer to stick with it, does not affect model size
attention_probs_dropout_prob = 0.1  # Tunable but safer to stick with it, does not affect model size
max_position_embeddings = 512  # We define our context length to be 32
type_vocab_size = 2  # Not very relevant and we ignored it...
initializer_range = 0.02  # Tunable but safer to stick with it, does not affect model size
layer_norm_eps = 1e-12  # Tunable but safer to stick with it, does not affect model size
pad_token_id = 0  # Our pad token id is defined by the tokenizer
position_embedding_type = 'absolute'  # Other choices exist but it's simpler
use_cache = True
classifier_dropout = None

batch_size_choices = [16, 32, 64]  # Not a hyperparameter for the model. We chose from some classic numbers

In [ ]:
# Final results
# These results correspond to our expectation. The model is about 100mb and is usable.

"""
embed_dim: 256
num_heads: 4
ff_dim: 128
num_layers: 2
batch_size: 32
"""

# Model training
* As can be seen in `./training/training_hf.ipynb`, we first tried training with `transformers` library, but we found the library a bit heavy... There were constant dependency issues and the library doesn't seem stable.
* We switched to simple pytorch training, but we still wrote some classes that resembles the `transformers` interface, like `Trainer` in `./training/trainer.py`
* We tested training for 2 epochs, in `./traning/training_pt.ipynb`. It went well. Each epoch takes about 30 mins. We then added some features to our custom `Trainer` api for better logging and visualization. The `training_pt.ipynb` is now deprecated, but since it records the first 2 epochs we can still have a look inside
* We wrote another script, `./training/training_continuous.ipynb`. For each time we train the model for 2 or 3 epochs and see how it behaves. As can be seen in the plot, the loss decreases stably as accuracy grows, but slower and slower over the epochs. After 17 epochs, the change is minimum.
* At the end it reached over 70% accuracy on the test set
* Overall, we trained the model for 19 epochs, which took about 10-15 hours.


## Commentary
* This is probably the sign of underfitting. We have about 3.7 million data and it is probable that the model is just not big enough to learn everything. At the end, we do have a accuracy over 70%, and we find that satisfying enough for a input method. Increasing model size is not necessarily a good choice in our case since it means more inference time, which can be annoying on a phone. We find that our model is a good compromise between performance and model size.
* It will be interesting to test other model architecture, like a decoder-only model, with similar size, on the same task. But we don't want to spend another 15 hours training a new model.
* We designed another `model2` in `./models/model_instancies.py` but we didn't train it

# Testing the model
* See `Pipeline` class in `./models/pipeline.py` and `test_model.ipynb`
* Two parameters are quite important in the `Pipeline.predict` method, namely `topk` and `filter_by_digit`.
    * `topk` allows model to output the number of candidates.
    * `filter_by_digit` is a boolean which lets us enable filtering based on user's t9 input. For example, if the context is `今天早餐我想吃543`, and the model predicts `鸡蛋（54326）` and `油条（9688126）`, `油条` will be filtered out since its t9 does not match user's input digits at all.
    * `topk` is a parameter that allows more candidates, and `filter_by_digits` filters out candidates that are clearly not intended by the user.
    *  We have to carefully choose the `topk` parameter since a small number might produce 0 candidates (all candidates are filtered out), and a large number is computationally expensive. After testing, we set `topk` to 500 and it worked generally well.
* Results are decent and we can type some simple sentences

# Interface
* See `./interface/README.md`
* Results are decent are allow us to type simple sentences from scratch like "今天早上我想吃鸡蛋". It struggles with more complicated sentences, with 成语, literature references, etc. We cannot type "唐僧是东土大唐来的高僧，他的徒弟是孙悟空". We also tried to type a few poems and it performs poorly.

# Conclusion and commentary
* We successfully implement a model for t9-hanzi prediction
* As shown by the `filter_by_digit` part, for this specific task, we cannot rely on deep learning method 100%.
* In our opinion, deeplearning method should be only responsible on suggesting a few possible candidates, and should be combined with traditional approaches.